# Lectura de paquetes y data

In [32]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import STL

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce verbosity de Optuna


# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils
from src.utils_ml import ml_plotting as plot_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

Utilizamos paths relativos para la lectura de la data

In [ ]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

In [ ]:
df_base.head(5)

In [ ]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [ ]:
df

# Preparación de la data

In [ ]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [ ]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [ ]:
df.shape

# EDA

## general

In [ ]:
df.isna().sum()

In [ ]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

## Tendencias

In [ ]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

In [ ]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


### Descomposición STL

In [ ]:
# graficamos la tendencia y estacionalidad de cada SKU usando STL
plot_utils.graficar_serie_con_descomposicion(df_sku, sku=sku, periodo=7)


# Feature engineering

## Variables temporales

In [ ]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

## Variables tipo lag

In [ ]:
df = fe_utils.create_lag_features(
    df, "pedidos", "sku", "fecha", max_daily_lag=14, weekday_lags=3
)
df.shape, df.columns

## Variables tipo promedio moviles

In [ ]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

## Variables lags de STL

In [ ]:
df = fe_utils.create_stl_features(
    df, "pedidos", "sku", "fecha", seasonal=7, stl_lags=14
)
df.shape, df.columns


## Aplanamiento de outliers en demanda

In [ ]:
df = fe_utils.cap_upper_outliers(df, "pedidos", "sku")

## Ajustes finales a la data

In [ ]:
# Eliminamos todas las filas con valores NaN para que no afecten el entrenamiento
df.dropna(inplace=True)
df.shape, df.columns

In [ ]:
df

# Modelling 

In [ ]:
## Preparación de data para modelling

# Separamos el conjunto de datos en entrenamiento y prueba, usando los últimos 7 días como prueba
test = df.tail(7)
df_mod = df[:-7].copy()

# Identificamos las variables predictoras
features = df_mod.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el numero de ventanas de evaluación y el tamaño de las ventanas
val_iter = 3
val_size = 7

## Evaluación modelos XGBoost

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Definimos el espacio de búsqueda de hiperparámetros para el modelo XGBoost
    # Usamos funciones lambda para que Optuna pueda sugerir valores
    param_grid = {
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 8, step=1),
        "learning_rate": lambda trial: trial.suggest_float(
            "learning_rate", 0.001, 0.1, step=0.001
        ),
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1500, step=50
        ),
        "subsample": lambda trial: trial.suggest_float(
            "subsample", 0.5, 1.0, step=0.02
        ),
        "colsample_bytree": lambda trial: trial.suggest_float(
            "colsample_bytree", 0.5, 1.0, step=0.02
        ),
        "gamma": lambda trial: trial.suggest_float("gamma", 1, 30, step=0.5),
        "reg_alpha": lambda trial: trial.suggest_float("reg_alpha", 1, 30, step=0.5),
        "reg_lambda": lambda trial: trial.suggest_float("reg_lambda", 1, 30, step=0.5),
        "min_child_weight": lambda trial: trial.suggest_int(
            "min_child_weight", 5, 20, step=1
        ),
        "random_state": 100,  # Fijamos la semilla para reproducibilidad
    }

    # Optimizamos el modelo XGBoost usando Optuna con ventana recursiva
    study = ml_utils.optimize_model_with_optuna(
        model_class=XGBRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=100,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Obtenemos el mejor trial del estudio
    # y almacenamos los resultados en un diccionario
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "XGBRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("\n-----------------------")


df_results_xgboost = pd.DataFrame(results)

In [ ]:
df_results_xgboost

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_xgboost[df_results_xgboost["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Random Forest

In [ ]:
results = []

for sku in df_mod["sku"].unique():
    print("\n-----------------------")
    print(f"🔍 Optimizando para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()

    # Tamaño de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para Random Forest
    param_grid = {
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1000, step=50
        ),
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 10, step=1),
        "min_samples_split": lambda trial: trial.suggest_int(
            "min_samples_split", 2, 10
        ),
        "min_samples_leaf": lambda trial: trial.suggest_int(
            "min_samples_leaf", 3, 15, step=1
        ),
        "max_features": lambda trial: trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        "bootstrap": lambda trial: trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Optimizamos usando tu función personalizada con modelo RandomForestRegressor
    study = ml_utils.optimize_model_with_optuna(
        model_class=RandomForestRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=100,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Registramos los resultados
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "RandomForestRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

# Creamos DataFrame con resultados
df_results_rf = pd.DataFrame(results)


In [ ]:
df_results_rf

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_rf[df_results_rf["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Elastic Net

In [34]:
results_elastic = []

for sku in df_mod["sku"].unique()[0:2]:
    print("\n-----------------------")
    print(f"🔍 Optimizando ElasticNet para SKU: {sku}\n")

    # Filtramos por SKU
    df_sku = df_mod[df_mod["sku"] == sku].copy()
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para ElasticNet
    param_grid = {
        "model__alpha": lambda trial: trial.suggest_float(
            "model__alpha", 0.0001, 1000.0, log=True
        ),
        "model__l1_ratio": lambda trial: trial.suggest_float(
            "model__l1_ratio", 0.0, 1.0, step=0.01
        ),  # 0 = Ridge, 1 = Lasso
        "model__random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Pipeline: escalado + modelo
    model_pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", ElasticNet(max_iter=20000))]
    )

    # Optimización con tu función
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: model_pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=100,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results_elastic.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "ElasticNet",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_elastic = pd.DataFrame(results_elastic)



-----------------------
🔍 Optimizando ElasticNet para SKU: SKU1



100%|██████████| 100/100 [00:15<00:00,  6.42it/s]



📌 Mejores Trials:
Trial 0 - SMAPE: 27.6900, GAP: 23.5400
Trial 1 - SMAPE: 28.0200, GAP: 23.2300
Trial 2 - SMAPE: 28.8100, GAP: 21.7800
Trial 3 - SMAPE: 27.7600, GAP: 23.4800
Trial 4 - SMAPE: 26.5600, GAP: 26.8300
Trial 5 - SMAPE: 25.5000, GAP: 29.2300
Trial 6 - SMAPE: 26.5000, GAP: 26.8900
Trial 7 - SMAPE: 28.1800, GAP: 22.7200
Trial 8 - SMAPE: 25.5900, GAP: 29.1000
Trial 9 - SMAPE: 28.7600, GAP: 21.8400
Trial 10 - SMAPE: 27.4400, GAP: 24.3300
Trial 11 - SMAPE: 27.2000, GAP: 24.5400
Trial 12 - SMAPE: 27.2700, GAP: 24.4800
Trial 13 - SMAPE: 25.5100, GAP: 29.2000
Trial 14 - SMAPE: 27.1600, GAP: 24.5800
Trial 15 - SMAPE: 26.7100, GAP: 26.4100
Trial 16 - SMAPE: 26.5600, GAP: 26.8300
Trial 17 - SMAPE: 28.4700, GAP: 22.5700
Trial 18 - SMAPE: 26.5400, GAP: 26.8500
Trial 19 - SMAPE: 28.1000, GAP: 23.1500
Trial 20 - SMAPE: 26.7700, GAP: 26.3700
Trial 21 - SMAPE: 25.8600, GAP: 28.7100
Trial 22 - SMAPE: 26.5300, GAP: 26.8600
Trial 23 - SMAPE: 28.9100, GAP: 21.6400
Trial 24 - SMAPE: 28.9100, GAP:

100%|██████████| 100/100 [00:13<00:00,  7.49it/s]


📌 Mejores Trials:
Trial 0 - SMAPE: 43.3000, GAP: 4.0700
Trial 1 - SMAPE: 43.3000, GAP: 4.0700
Trial 2 - SMAPE: 43.3000, GAP: 4.0700
Trial 3 - SMAPE: 28.1500, GAP: 5.4900
Trial 4 - SMAPE: 42.3000, GAP: 4.5900
Trial 5 - SMAPE: 43.0100, GAP: 4.2200
Trial 6 - SMAPE: 28.1900, GAP: 5.1000
Trial 7 - SMAPE: 43.1800, GAP: 4.1300
Trial 8 - SMAPE: 42.8500, GAP: 4.3200
Trial 9 - SMAPE: 43.3000, GAP: 4.0700
Trial 10 - SMAPE: 43.3000, GAP: 4.0700
Trial 11 - SMAPE: 42.2600, GAP: 4.6300
Trial 12 - SMAPE: 42.9100, GAP: 4.2700
Trial 13 - SMAPE: 41.8700, GAP: 4.8400
Trial 14 - SMAPE: 43.2100, GAP: 4.1200
Trial 15 - SMAPE: 43.0700, GAP: 4.1900
Trial 16 - SMAPE: 42.1500, GAP: 4.7000
Trial 17 - SMAPE: 28.1600, GAP: 5.4300
Trial 18 - SMAPE: 28.1500, GAP: 5.4900
Trial 19 - SMAPE: 28.1800, GAP: 5.4200
Trial 20 - SMAPE: 42.8100, GAP: 4.3300
Trial 21 - SMAPE: 42.0100, GAP: 4.7500

🏆 Best Trial Params:
{'model__alpha': 413.0054756664381, 'model__l1_ratio': 0.78}
-----------------------


## Selección mejor modelo y ajuste final

In [ ]:
# Union de resultados de los modelos
df_results = pd.concat(
    [df_results_xgboost, df_results_rf, df_results_elastic],
    ignore_index=True,
)

# Por cada SKU, obtenemos el mejor modelo
df_best_models = df_results.loc[
    df_results.groupby("sku")["best_smape"].idxmin()
].reset_index(drop=True)


# Ordenamos por mejor SMAPE
df_best_models = df_best_models.sort_values(by="best_smape", ascending=True)

df_best_models

In [ ]:
# Reentrenamos la serie de tiempo de cada SKU con el mejor modelo, usando el conjunto de entrenamiento completo



# Predicción y graficas 